In [ ]:
#1
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import json

# =====================================================================
# TRẠM 1: ĐỌC DỮ LIỆU PARQUET & KHUẾCH ĐẠI HẠT GIỐNG (BOOTSTRAPPING)
# =====================================================================

print("🚀 ĐANG KHỞI ĐỘNG ĐỘNG CƠ BIG DATA (APACHE SPARK)...")
spark = SparkSession.builder \
    .appName("Music_Taste_Engine_Module2") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.default.parallelism", "400") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.local.dir", "/kaggle/working/spark_tmp") \
    .getOrCreate()

# 1. ĐỌC THẲNG THƯ MỤC PARQUET (Bỏ qua toàn bộ Regex & Explode vì data đã sạch)
print("⏳ Đang nạp dữ liệu sạch từ thư mục Parquet...")
path_parquet = "/kaggle/input/datasets/terrobstark/supercleandata/CLEANED_DATASET_PARQUET"

clean_df = spark.read.parquet(path_parquet) \
    .select("user_id", "artist_name") \
    .filter(F.col("user_id").isNotNull()) \
    .filter(F.col("artist_name").isNotNull() & (F.col("artist_name") != "")) \
    .repartition(400, "user_id") \
    .persist()

print(f"✅ Đã tải thành công dữ liệu! Tổng số dòng: {clean_df.count():,}")

# 2. NÉN DỮ LIỆU THÀNH GIỎ HÀNG (SET) THEO USER
print("⏳ Đang nén dữ liệu thành Tập hợp (Set) theo User...")
user_artists_df = clean_df.groupBy("user_id").agg(
    F.collect_set("artist_name").alias("artist_set")
)

filtered_users_df = user_artists_df.filter(
    (F.size(F.col("artist_set")) >= 5) & 
    (F.size(F.col("artist_set")) <= 1000)
).persist()
print("✅ Đã nén thành Giỏ xong!")

# =====================================================================
# KHUẾCH ĐẠI HẠT GIỐNG (BOOTSTRAPPING - Giữ nguyên logic PMI của bạn)
# =====================================================================
print("🚀 Đang kích hoạt Bootstrapping...")

seed_genres_f0 = {
    "POP": ["taylor swift", "ariana grande", "justin bieber", "ed sheeran"],
    "HIPHOP": ["eminem", "kendrick lamar", "drake", "travis scott"],
    "EDM": ["martin garrix", "avicii", "calvin harris", "skrillex"],
    "RNB": ["the weeknd", "sza", "frank ocean", "bruno mars"],
    "ROCK": ["linkin park", "queen", "ac/dc", "coldplay"]
}

seed_data = [(artist, genre) for genre, artists in seed_genres_f0.items() for artist in artists]
seed_df = spark.createDataFrame(seed_data, ["seed_artist", "genre"])

user_exposure = clean_df.join(
    seed_df,
    clean_df.artist_name == seed_df.seed_artist,
    "inner"
).select("user_id", "genre").distinct()

user_genre_counts = user_exposure.groupBy("user_id").count()
pure_users = user_genre_counts.filter(F.col("count") == 1).select("user_id")
user_exposure_pure = user_exposure.join(pure_users, "user_id", "inner")

expanded_candidates = user_exposure_pure.join(clean_df, "user_id", "inner")
candidate_counts = expanded_candidates.groupBy("genre", "artist_name") \
    .count().withColumnRenamed("count", "genre_count")

global_counts = clean_df.groupBy("artist_name") \
    .count().withColumnRenamed("count", "global_count")

MIN_GENRE_COUNT = 20
MIN_GLOBAL_COUNT = 50

# Tính tổng số cặp user-artist trên toàn hệ thống
total_all = clean_df.count()

# Tính tổng lượt nghe của từng thể loại (genre)
genre_totals = candidate_counts.groupBy("genre").agg(F.sum("genre_count").alias("total_genre"))

# Tính PMI để làm relevance_score
scored_candidates = candidate_counts.join(global_counts, "artist_name") \
    .join(genre_totals, "genre") \
    .withColumn("p_artist_given_genre", F.col("genre_count") / F.col("total_genre")) \
    .withColumn("p_artist", F.col("global_count") / total_all) \
    .withColumn("relevance_score", F.log2(F.col("p_artist_given_genre") / F.col("p_artist"))) \
    .filter(F.col("genre_count") >= MIN_GENRE_COUNT) \
    .filter(F.col("global_count") >= MIN_GLOBAL_COUNT)

f0_artist_list = [artist for artists in seed_genres_f0.values() for artist in artists]
new_candidates = scored_candidates.filter(~F.col("artist_name").isin(f0_artist_list))

window_spec = Window.partitionBy("genre").orderBy(
    F.col("relevance_score").desc(),
    F.col("genre_count").desc()
)
top_f1_artists = new_candidates.withColumn("rank", F.row_number().over(window_spec)) \
                               .filter(F.col("rank") <= 50) \
                               .select("genre", "artist_name")

f1_seed_df = top_f1_artists.groupBy("genre").agg(F.collect_list("artist_name").alias("f1_artists"))
f1_seed_dict = {row['genre']: row['f1_artists'] for row in f1_seed_df.collect()}

final_seeds = {}
print("\n🌟 KẾT QUẢ KHUẾCH ĐẠI HẠT GIỐNG:")
for genre in seed_genres_f0.keys():
    combined = seed_genres_f0[genre] + f1_seed_dict.get(genre, [])
    final_seeds[genre] = combined
    print(f"   => Nhóm {genre}: {len(combined)} Hạt giống (VD mới: {combined[len(seed_genres_f0[genre]):len(seed_genres_f0[genre])+4]}...)")

print("\n🎉 HOÀN TẤT TRẠM 1!")

# Lưu checkpoint dạng Parquet
filtered_users_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_filtered_users")
clean_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_clean_df")
with open("/kaggle/working/ckpt_final_seeds.json", "w") as f:
    json.dump(final_seeds, f)
print("💾 Đã lưu checkpoint Trạm 1!")

In [ ]:
#2
from pyspark.ml.feature import CountVectorizer, MinHashLSH
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

print("\n🚀 BẮT ĐẦU TRẠM 2: CHUYỂN ĐỔI SANG VECTOR VÀ BĂM DỮ LIỆU LSH...")

cv = CountVectorizer(inputCol="artist_set", outputCol="features", minDF=3)
print("⏳ Đang Vector hóa dữ liệu...")
cv_model = cv.fit(filtered_users_df)
vectorized_df = cv_model.transform(filtered_users_df)

@F.udf(returnType=IntegerType())
def count_nonzeros(v):
    return v.numNonzeros()

valid_vectorized_df = vectorized_df.filter(count_nonzeros(F.col("features")) > 0).cache()
print("✅ Đã dọn dẹp các Vector rỗng!")

mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
print("⏳ Đang tạo Chữ ký số (Signature)...")
mh_model = mh.fit(valid_vectorized_df)
users_with_hashes = mh_model.transform(valid_vectorized_df)

print("⏳ Đang kích hoạt LSH để nối Cạnh Đồ thị...")
threshold = 0.5  
edges_df = mh_model.approxSimilarityJoin(
    users_with_hashes, users_with_hashes, threshold, distCol="JaccardDistance"
).select(
    F.col("datasetA.user_id").alias("src"),
    F.col("datasetB.user_id").alias("dst"),
    (1 - F.col("JaccardDistance")).alias("weight")
)

final_edges_df = edges_df.filter(F.col("src") < F.col("dst")).persist()

# Dùng limit().toPandas() thay show() tránh OOM
print("✅ Đã dựng xong Mạng lưới! 5 Cạnh đầu tiên:")
print(final_edges_df.limit(5).toPandas().to_string(index=False))

# Lưu checkpoint dạng Parquet
final_edges_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_edges")
print("💾 Đã lưu checkpoint Trạm 2!")
print("\n🎉 HOÀN TẤT TRẠM 2!")
cv_model.write().overwrite().save("/kaggle/working/cv_model_saved")
mh_model.write().overwrite().save("/kaggle/working/mh_model_saved")
print("💾 Đã lưu Model CV và LSH thành công!")

In [ ]:
#3
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType, ArrayType
from pyspark.sql.window import Window
from functools import reduce

print("\n🚀 BẮT ĐẦU TRẠM 3: LABEL PROPAGATION QUA ĐỒ THỊ...")

spark.sparkContext.setCheckpointDir("/kaggle/working/checkpoints")

GENRES = ["POP", "HIPHOP", "EDM", "RNB", "ROCK"]

# BƯỚC 1: GẮN NHÃN HẠT GIỐNG (TỐI ƯU DOMINANT SEED)
print("⏳ Đang gắn nhãn Hạt giống cho các Node...")

# 1. Đưa dictionary hạt giống về DataFrame
seed_data = [(genre, artist) for genre, artists in final_seeds.items() for artist in artists]
seeds_df = spark.createDataFrame(seed_data, ["seed_genre", "seed_artist_name"])

# 2. Tìm các user có nghe nghệ sĩ hạt giống
user_seed_matches = clean_df.join(
    seeds_df, 
    clean_df.artist_name == seeds_df.seed_artist_name, 
    "inner"
)

# 3. Đếm số lượng nghệ sĩ hạt giống theo từng thể loại mà user đã nghe
user_seed_counts = user_seed_matches.groupBy("user_id", "seed_genre").count()

# 4. Lấy thể loại hạt giống xuất hiện nhiều nhất (Dominant) làm nhãn gốc
window_spec = Window.partitionBy("user_id").orderBy(F.col("count").desc())
seed_labels_pure_df = user_seed_counts.withColumn("rank", F.row_number().over(window_spec)) \
                                      .filter(F.col("rank") == 1) \
                                      .select("user_id", "seed_genre") \
                                      .persist()

print(f"✅ Số User được gắn nhãn Hạt giống: {seed_labels_pure_df.count():,}")

# --- CHIA TẬP TRAIN/TEST (HOLD-OUT) ---
print("⏳ Đang chia tập Train/Test để chuẩn bị đánh giá...")
train_seeds, test_seeds = seed_labels_pure_df.randomSplit([0.8, 0.2], seed=42)
test_seeds.write.mode("overwrite").parquet("/kaggle/working/ckpt_test_seeds")
print(f"✅ Đã cất đi {test_seeds.count():,} users để test.")
# BƯỚC 2: KHỞI TẠO VECTOR XÁC SUẤT
print("⏳ Đang khởi tạo Vector xác suất ban đầu...")

all_node_ids = final_edges_df.select(F.col("src").alias("user_id")) \
    .union(final_edges_df.select(F.col("dst").alias("user_id"))) \
    .distinct()

# LƯU Ý: Chỉ join với train_seeds, những user trong test_seeds sẽ coi như chưa có nhãn
nodes_with_seed = all_node_ids.join(train_seeds, "user_id", "left")
def init_prob_vector(genre):
    if genre is None:
        return [0.2, 0.2, 0.2, 0.2, 0.2]
    vec = [0.0] * 5
    vec[GENRES.index(genre)] = 1.0
    return vec

init_udf = F.udf(init_prob_vector, ArrayType(FloatType()))

current_nodes = nodes_with_seed \
    .withColumn("prob_vector", init_udf(F.col("seed_genre"))) \
    .withColumn("is_seed", F.col("seed_genre").isNotNull()) \
    .select("user_id", "prob_vector", "is_seed", "seed_genre") \
    .checkpoint()
print("✅ Đã khởi tạo Vector xác suất!")

# BƯỚC 3: LAN TRUYỀN NHÃN
print("⏳ Đang lan truyền Nhãn qua Đồ thị...")

edges_bidirectional = final_edges_df.select(
    F.col("src").alias("user_id"),
    F.col("dst").alias("neighbor_id"),
    F.col("weight")
).union(
    final_edges_df.select(
        F.col("dst").alias("user_id"),
        F.col("src").alias("neighbor_id"),
        F.col("weight")
    )
).persist()

ALPHA = 0.7
MAX_ITERATIONS = 15
EPSILON = 1e-4

iteration = 0
delta = float('inf')

while delta > EPSILON and iteration < MAX_ITERATIONS:
    # Lưu vector cũ để so sánh sự thay đổi
    old_nodes = current_nodes.select("user_id", F.col("prob_vector").alias("old_vector"))
    
    neighbor_vectors = edges_bidirectional.join(
        current_nodes.select("user_id", "prob_vector"),
        edges_bidirectional.neighbor_id == current_nodes.user_id,
        "inner"
    ).select(
        edges_bidirectional.user_id,
        F.col("weight"),
        F.col("prob_vector").alias("neighbor_prob")
    )

    agg_df = neighbor_vectors.groupBy("user_id").agg(
        F.sum("weight").alias("total_weight"),
        *[F.sum(F.col("neighbor_prob")[i] * F.col("weight")).alias(f"sum_g{i}") for i in range(5)]
    )

    neighbor_mean_df = agg_df.withColumn(
        "neighbor_mean",
        F.array(*[F.col(f"sum_g{i}") / F.col("total_weight") for i in range(5)])
    ).select("user_id", "neighbor_mean")

    updated = current_nodes.join(neighbor_mean_df, "user_id", "left")

    def blend_vectors(is_seed, prob_vec, neighbor_mean):
        if neighbor_mean is None:
            return prob_vec
        if is_seed:
            return [ALPHA * n + (1 - ALPHA) * p for p, n in zip(prob_vec, neighbor_mean)]
        else:
            return list(neighbor_mean)

    blend_udf = F.udf(blend_vectors, ArrayType(FloatType()))

    current_nodes = updated.withColumn(
        "prob_vector",
        blend_udf(F.col("is_seed"), F.col("prob_vector"), F.col("neighbor_mean"))
    ).select("user_id", "prob_vector", "is_seed", "seed_genre") \
     .localCheckpoint()

    # Tính độ lệch (Delta) để kiểm tra hội tụ
    diff_df = current_nodes.join(old_nodes, "user_id").withColumn(
        "max_diff",
        F.greatest(*[F.abs(F.col("prob_vector")[i] - F.col("old_vector")[i]) for i in range(5)])
    )
    delta = diff_df.agg(F.max("max_diff")).collect()[0][0]

    iteration += 1
    print(f"   ✅ Vòng {iteration}: Delta = {delta:.6f}")

print("✅ Lan truyền xong!")

# BƯỚC 4: XUẤT KẾT QUẢ
print("⏳ Đang đóng gói Kết quả...")

result_df = current_nodes.select(
    "user_id",
    *[F.round(F.col("prob_vector")[i] * 100, 1).alias(f"{GENRES[i]}_%") for i in range(5)]
)

result_df = result_df.withColumn(
    "max_val", F.greatest(*[F.col(f"{g}_%") for g in GENRES])
).withColumn(
    "Dominant_Genre",
    F.when(F.col("POP_%") == F.col("max_val"), "POP")
     .when(F.col("HIPHOP_%") == F.col("max_val"), "HIPHOP")
     .when(F.col("EDM_%") == F.col("max_val"), "EDM")
     .when(F.col("RNB_%") == F.col("max_val"), "RNB")
     .otherwise("ROCK")
).drop("max_val")

user_taste_profile_df = result_df.persist()

print("\n🌟 MẪU KẾT QUẢ USER TASTE PROFILE:")
print(user_taste_profile_df.limit(10).toPandas().to_string(index=False))

print(f"\n📊 Thống kê Dominant Genre:")
print(user_taste_profile_df.groupBy("Dominant_Genre").count()
      .orderBy(F.col("count").desc()).toPandas().to_string(index=False))

# Lưu checkpoint dạng Parquet
user_taste_profile_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_user_taste")
print("💾 Đã lưu checkpoint Trạm 3!")
print("\n🎉 HOÀN TẤT TRẠM 3!")

In [ ]:
#4
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd

print("\n🚀 BẮT ĐẦU TRẠM 4: ĐÓNG GÓI DỮ LIỆU ĐẦU RA...")

GENRES = ["POP", "HIPHOP", "EDM", "RNB", "ROCK"]

# BƯỚC 0: BỔ SUNG NHÃN CHO USER NGOÀI ĐỒ THỊ
print("⏳ Bổ sung nhãn cho User ngoài đồ thị...")

users_in_graph = user_taste_profile_df.select("user_id")
all_users = filtered_users_df.select("user_id")
users_outside_graph = all_users.join(users_in_graph, "user_id", "left_anti")

# ĐÃ SỬA THÀNH seed_labels_pure_df
outside_with_seed = users_outside_graph.join(seed_labels_pure_df, "user_id", "inner")

outside_genre_counts = outside_with_seed.groupBy("user_id", "seed_genre").count()
outside_total = outside_genre_counts.groupBy("user_id") \
    .agg(F.sum("count").alias("total_seeds"))
outside_ratios = outside_genre_counts.join(outside_total, "user_id") \
    .withColumn("ratio", F.round((F.col("count") / F.col("total_seeds")) * 100, 1))

outside_profiles = outside_ratios.groupBy("user_id").pivot(
    "seed_genre", GENRES
).agg(F.first("ratio")).fillna(0.0)

for g in GENRES:
    if g in outside_profiles.columns:
        outside_profiles = outside_profiles.withColumnRenamed(g, f"{g}_%")
    else:
        outside_profiles = outside_profiles.withColumn(f"{g}_%", F.lit(0.0))

outside_profiles = outside_profiles.withColumn(
    "max_val", F.greatest(*[F.col(f"{g}_%") for g in GENRES])
).withColumn(
    "Dominant_Persona",
    F.when(F.col("POP_%") == F.col("max_val"), "POP")
     .when(F.col("HIPHOP_%") == F.col("max_val"), "HIPHOP")
     .when(F.col("EDM_%") == F.col("max_val"), "EDM")
     .when(F.col("RNB_%") == F.col("max_val"), "RNB")
     .otherwise("ROCK")
).withColumn(
    "lsh_bucket_id", F.md5(F.col("user_id").cast("string"))
).drop("max_val")

print(f"✅ User ngoài đồ thị có seed: {outside_profiles.count():,}")

# BƯỚC 1: GỘP PROFILE LPA + OUTSIDE
user_taste_final = user_taste_profile_df \
    .withColumnRenamed("Dominant_Genre", "Dominant_Persona") \
    .join(
        users_with_hashes.select(
            "user_id",
            F.md5(F.col("user_id").cast("string")).alias("lsh_bucket_id")
        ), "user_id", "left"
    )

user_taste_combined = user_taste_final.union(
    outside_profiles.select(user_taste_final.columns)
).persist()

# ĐẦU RA 1: ARTIST GENRE PROFILE
print("\n⏳ Đang tạo Bảng 1: Artist Genre Profile...")

user_taste_labeled = user_taste_combined.filter(F.col("POP_%") != 20.0)

artist_fan_profile = clean_df.join(
    user_taste_labeled.select("user_id", *[f"{g}_%" for g in GENRES]),
    "user_id", "inner"
)

artist_genre_profile = artist_fan_profile.groupBy("artist_name").agg(
    F.count("user_id").alias("fan_count"),
    *[F.round(F.avg(f"{g}_%"), 1).alias(f"{g}_%") for g in GENRES]
).filter(F.col("fan_count") >= 10)

artist_genre_profile = artist_genre_profile.withColumn(
    "max_val", F.greatest(*[F.col(f"{g}_%") for g in GENRES])
).withColumn(
    "Dominant_Genre",
    F.when(F.col("POP_%") == F.col("max_val"), "POP")
     .when(F.col("HIPHOP_%") == F.col("max_val"), "HIPHOP")
     .when(F.col("EDM_%") == F.col("max_val"), "EDM")
     .when(F.col("RNB_%") == F.col("max_val"), "RNB")
     .otherwise("ROCK")
).drop("max_val")

print("\n⏳ Đang lưu Bảng 1: Artist Genre Profile (Thuần Spark)...")
artist_genre_profile.orderBy(F.col("fan_count").desc()) \
    .coalesce(1).write.mode("overwrite").option("header", "true") \
    .csv("/kaggle/working/artist_genre_profile_spark")
print("✅ Đã lưu Artist Genre Profile!")

print("\n⏳ Đang lưu Bảng 2: User Taste Profile (Thuần Spark)...")
user_taste_combined.coalesce(1).write.mode("overwrite").option("header", "true") \
    .csv("/kaggle/working/user_taste_profile_spark")
print("✅ Đã lưu User Taste Profile!")

# Đếm số lượng bằng lệnh .count() của Spark thay vì len() của Pandas
print(f"✅ User Taste Profile: {user_taste_combined.count():,} users")

print("\n" + "="*60)
print("📊 TỔNG KẾT TRẠM 4:")
print(f"   - Số Nghệ sĩ đã phân tích : {artist_genre_profile.count():,}")
print(f"   - Số User đã phân tích     : {user_taste_combined.count():,}")
print(f"   - File xuất ra             : /kaggle/working/ (Dạng thư mục phân tán)")
print("="*60)
print("\n🎉 HOÀN TẤT TRẠM 4! Dữ liệu đã an toàn cập bến.")

In [ ]:
#5
import pyspark.sql.functions as F
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.types import DoubleType
import json

print("\n🚀 BẮT ĐẦU TRẠM 5: ĐÁNH GIÁ CHUYÊN SÂU (100% PYSPARK)...")

test_seeds_df = spark.read.parquet("/kaggle/working/ckpt_test_seeds")

eval_df = user_taste_profile_df.join(
    test_seeds_df.select("user_id", F.col("seed_genre").alias("True_Label")),
    "user_id", "inner"
).select("user_id", "True_Label", "Dominant_Genre").persist()

total_test = eval_df.count()
print(f"✅ Đã tải tập Test ({total_test:,} users).")

print("\n📊 MA TRẬN NHẦM LẪN (Crosstab):")
eval_df.crosstab("True_Label", "Dominant_Genre").orderBy("True_Label_Dominant_Genre").show()

GENRES = ["POP", "HIPHOP", "EDM", "RNB", "ROCK"]
mapping = {g: float(i) for i, g in enumerate(GENRES)}
mapping_expr = F.create_map([F.lit(x) for k, v in mapping.items() for x in (k, v)])

score_df = eval_df.withColumn("label", mapping_expr.getItem(F.col("True_Label")).cast(DoubleType())) \
                  .withColumn("prediction", mapping_expr.getItem(F.col("Dominant_Genre")).cast(DoubleType()))

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

accuracy = evaluator_acc.evaluate(score_df)
macro_f1 = evaluator_f1.evaluate(score_df)

print("\n📈 BÁO CÁO ĐÁNH GIÁ MÔ HÌNH:")
print("="*60)
print(f"   Độ chính xác (Accuracy)  : {accuracy * 100:.2f}%")
print(f"   Chỉ số F1 (Macro F1)     : {macro_f1 * 100:.2f}%")
print("="*60)
print("\n🎉 HOÀN TẤT ĐÁNH GIÁ! KIẾN TRÚC ĐÃ ĐẠT CHUẨN BIG DATA.")

eval_results = {
    "evaluation_method": "Hold-out Validation (Native PySpark)",
    "accuracy": round(accuracy, 4),
    "f1_score": round(macro_f1, 4),
    "total_test_users": total_test
}
with open("/kaggle/working/evaluation_results.json", "w") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)